In [1]:
# ==========================================================
# Generate Realistic Synthetic Clinical Dataset for EyePACS
# ==========================================================

import os
import random
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
DATASET_PATH = r"C:\Users\VICTUS\Downloads\Train_Dataset_DR"
OUTPUT_DIR = r"D:\Diabetic Retinopathy Prediction\data"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "clinical_data.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)
np.random.seed(42)

# ----------------------------------------------------------
# Helper Function
# ----------------------------------------------------------
def clip(x, low, high):
    return max(low, min(high, x))

# ----------------------------------------------------------
# Generate Dataset
# ----------------------------------------------------------
records = []

for grade in range(5):

    folder = os.path.join(DATASET_PATH, str(grade))

    if not os.path.exists(folder):
        continue

    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for image in images:

        # -----------------------------
        # Latent disease severity
        # -----------------------------
        severity = np.random.normal(grade, 0.9)
        severity = clip(severity, 0, 4)

        # -----------------------------
        # Age
        # -----------------------------
        age = int(np.random.normal(52 + severity*1.8, 11))
        age = clip(age, 30, 80)

        # -----------------------------
        # Diabetes duration
        # -----------------------------
        duration = np.random.normal(7 + severity*2.0, 5)
        duration = clip(duration, 0, 30)

        # -----------------------------
        # HbA1c
        # -----------------------------
        hba1c = np.random.normal(
            6.7 + severity*0.35 + duration*0.03,
            0.9
        )
        hba1c = round(clip(hba1c, 5.2, 12.5),1)

        # -----------------------------
        # Fasting glucose
        # (depends mostly on HbA1c)
        # -----------------------------
        glucose = np.random.normal(
            18*hba1c,
            22
        )
        glucose = int(clip(glucose,70,320))

        # -----------------------------
        # BMI
        # -----------------------------
        bmi = np.random.normal(
            25.5 + severity*0.4,
            3.2
        )
        bmi = round(clip(bmi,18,40),1)

        # -----------------------------
        # Blood pressure
        # -----------------------------
        sbp = np.random.normal(
            120 + 0.45*age + 0.3*(bmi-25),
            10
        )
        sbp = int(clip(sbp,95,190))

        dbp = np.random.normal(
            75 + 0.20*(sbp-120),
            7
        )
        dbp = int(clip(dbp,60,110))

        # -----------------------------
        # Cholesterol
        # -----------------------------
        cholesterol = np.random.normal(
            180 + (bmi-25)*2 + severity*5,
            18
        )
        cholesterol = int(clip(cholesterol,120,320))

        # -----------------------------
        # Creatinine
        # -----------------------------
        creatinine = np.random.normal(
            0.8 + duration*0.02,
            0.15
        )
        creatinine = round(clip(creatinine,0.5,2.5),2)

        # -----------------------------
        # Gender
        # -----------------------------
        gender = random.choice(["Male","Female"])

        # -----------------------------
        # Smoking
        # (depends weakly on age)
        # -----------------------------
        if age < 45:
            smoking = np.random.choice(
                ["Never","Former","Current"],
                p=[0.70,0.15,0.15]
            )
        elif age < 60:
            smoking = np.random.choice(
                ["Never","Former","Current"],
                p=[0.55,0.25,0.20]
            )
        else:
            smoking = np.random.choice(
                ["Never","Former","Current"],
                p=[0.45,0.35,0.20]
            )

        # -----------------------------
        # Hypertension
        # (depends on age + BMI)
        # -----------------------------
        risk = 0.15

        if age > 55:
            risk += 0.20

        if bmi > 30:
            risk += 0.15

        if duration > 15:
            risk += 0.10

        risk = min(risk,0.70)

        hypertension = np.random.choice(
            ["No","Yes"],
            p=[1-risk,risk]
        )

        records.append({

            "Image_ID": image,

            "Image_Path": os.path.join(folder,image),

            "DR_Grade": grade,

            "Age": age,

            "Gender": gender,

            "Duration_DM_Years": round(duration,1),

            "HbA1c": hba1c,

            "Fasting_Glucose_mg_dL": glucose,

            "BMI": bmi,

            "Systolic_BP": sbp,

            "Diastolic_BP": dbp,

            "Smoking_Status": smoking,

            "Hypertension": hypertension,

            "Total_Cholesterol_mg_dL": cholesterol,

            "Serum_Creatinine_mg_dL": creatinine

        })

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------

clinical_df = pd.DataFrame(records)

clinical_df.to_csv(OUTPUT_CSV,index=False)

print("="*60)
print("Clinical dataset generated successfully")
print("="*60)

print("Total Samples :",len(clinical_df))
print("Saved File    :",OUTPUT_CSV)

print("\nClass Distribution")
print(clinical_df["DR_Grade"].value_counts().sort_index())

print("\nCorrelation with DR Grade")
corr = clinical_df.corr(numeric_only=True)["DR_Grade"]
print(corr.sort_values(ascending=False))

print("\nFirst 5 Samples")
print(clinical_df.head())

Clinical dataset generated successfully
Total Samples : 108227
Saved File    : D:\Diabetic Retinopathy Prediction\data\clinical_data.csv

Class Distribution
DR_Grade
0    55178
1    18470
2    24214
3      890
4     9475
Name: count, dtype: int64

Correlation with DR Grade
DR_Grade                   1.000000
HbA1c                      0.401357
Duration_DM_Years          0.370889
Total_Cholesterol_mg_dL    0.288783
Fasting_Glucose_mg_dL      0.257766
Serum_Creatinine_mg_dL     0.220245
Age                        0.162215
BMI                        0.127033
Systolic_BP                0.086492
Diastolic_BP               0.024964
Name: DR_Grade, dtype: float64

First 5 Samples
               Image_ID                                         Image_Path  \
0  005b95c28852-600.jpg  C:\Users\VICTUS\Downloads\Train_Dataset_DR\0\0...   
1  0097f532ac9f-600.jpg  C:\Users\VICTUS\Downloads\Train_Dataset_DR\0\0...   
2  00cc2b75cddd-600.jpg  C:\Users\VICTUS\Downloads\Train_Dataset_DR\0\0...   
3  00f